# Polypharmacy Pipeline -- Resume From Cache (Section 5-7)
### Interpretability, delirium/tabular leakage fix, and final export

This notebook **resumes** the INDICon pipeline from cached artifacts produced by
the local run (`notebooks/run_pipeline_local.py`) -- it does **not** re-pull
`prescriptions`/`labevents`/`chartevents`/`inputevents` from BigQuery, does not
rebuild the co-administration graph, and does not retrain node2vec. That work is
already done and cached. This notebook only needs one small BigQuery query
(comorbidity counts from `diagnoses_icd`, filtered to the ~10k cached hadm_ids)
to rebuild the tabular feature matrix, since that one piece was never persisted.

**Runtime**: select `Runtime > Change runtime type > T4 GPU` before running.
Being upfront about what the GPU actually buys you here: sklearn's
LogisticRegression/RandomForest/MLPClassifier and node2vec's gensim backend are
CPU-only regardless of GPU -- a T4 will NOT speed those up. XGBoost can use the
GPU (`device="cuda"`), which this notebook enables automatically when available.
The much bigger speedup vs. the local run comes from a different fix: the
interpretability perturbation analysis previously called `model.predict_proba()`
once per drug per high-risk patient (tens of thousands of individual calls) --
this version batches every perturbation for an entire outcome into a single
vectorized `predict_proba()` call, which is what actually removes the
bottleneck you hit locally.

## Files to upload (via the file-upload cell in Section 1)
Upload these 5 files from your local `results/` folder:
- `cohort.csv`
- `prescriptions_normalized.csv`
- `population_graphs.pkl`
- `node2vec_embeddings.pkl`
- `model_results.csv`


## 1. Setup, auth, and file upload

In [ ]:
!pip install google-cloud-bigquery google-cloud-bigquery-storage pandas networkx scikit-learn xgboost shap tqdm db-dtypes -q


In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = "unified-v2-494419"  # same GCP project used in the original run
client = bigquery.Client(project=PROJECT_ID)

print(f"BigQuery client initialized for billing project: {PROJECT_ID}")


In [ ]:
import os
import json
import pickle
import zipfile
import warnings
import subprocess
from collections import defaultdict
from itertools import combinations
from datetime import datetime

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

MANIFEST = []

def save_manifest_entry(path, description):
    MANIFEST.append({"file": path, "description": description})

# GPU detection -- only XGBoost benefits; sklearn/gensim stay CPU-only regardless.
try:
    subprocess.check_output("nvidia-smi")
    XGB_DEVICE = "cuda"
    print("GPU detected -- XGBoost will use device='cuda'.")
except Exception:
    XGB_DEVICE = "cpu"
    print("No GPU detected -- XGBoost will use device='cpu'.")


In [ ]:
# --- Upload the 5 cached files listed above ---
from google.colab import files

print("Select: cohort.csv, prescriptions_normalized.csv, population_graphs.pkl, "
      "node2vec_embeddings.pkl, model_results.csv")
uploaded = files.upload()

REQUIRED_FILES = [
    "cohort.csv", "prescriptions_normalized.csv", "population_graphs.pkl",
    "node2vec_embeddings.pkl", "model_results.csv",
]
missing = [f for f in REQUIRED_FILES if f not in uploaded]
if missing:
    raise RuntimeError(f"Missing required uploaded files: {missing}")
print("All required files uploaded successfully.")


## 2. Reload cached cohort, prescriptions, graph, and embeddings

No BigQuery calls in this section -- everything here comes from the uploaded files.


In [ ]:
cohort_df = pd.read_csv("cohort.csv")
rx_df = pd.read_csv("prescriptions_normalized.csv")

hadm_drugs = rx_df.groupby("hadm_id")["generic_drug"].apply(lambda s: sorted(set(s)))

print(f"cohort_df: {cohort_df.shape}")
print(f"rx_df: {rx_df.shape}")
print(f"hadm_drugs: {len(hadm_drugs)} admissions")

OUTCOMES = ["aki_label", "delirium_label", "bleeding_label"]
for outcome in OUTCOMES:
    print(f"{outcome}: {cohort_df[outcome].sum()} / {len(cohort_df)} positive")


In [ ]:
with open("population_graphs.pkl", "rb") as f:
    graph_bundle = pickle.load(f)

G_weighted = graph_bundle["G_weighted"]
G_unweighted = graph_bundle["G_unweighted"]
per_stay_graphs = graph_bundle["per_stay_graphs"]

print(f"G_weighted: {G_weighted.number_of_nodes()} nodes, {G_weighted.number_of_edges()} edges")
print(f"per_stay_graphs: {len(per_stay_graphs)} stays")


In [ ]:
with open("node2vec_embeddings.pkl", "rb") as f:
    emb_bundle = pickle.load(f)

drug_embeddings = emb_bundle["embeddings"]
N2V_HYPERPARAMS = emb_bundle["hyperparameters"]
N2V_DIMENSIONS = N2V_HYPERPARAMS["dimensions"]

print(f"Loaded node2vec embeddings for {len(drug_embeddings)} drugs, dim={N2V_DIMENSIONS}")
print(f"Hyperparameters: {N2V_HYPERPARAMS}")


## 3. Rebuild patient-level features (embedding + graph-theoretic; no BigQuery needed)

In [ ]:
def patient_embedding_feature(drugs):
    vecs = [drug_embeddings[d] for d in drugs if d in drug_embeddings]
    if not vecs:
        return np.zeros(N2V_DIMENSIONS)
    return np.mean(vecs, axis=0)

def patient_graph_theoretic_features(hadm_id, drugs):
    weights = []
    for u, v in combinations(drugs, 2):
        if G_weighted.has_edge(u, v):
            weights.append(G_weighted[u][v]["weight"])
    subG = per_stay_graphs.get(hadm_id, nx.Graph())
    density = nx.density(subG) if subG.number_of_nodes() > 1 else 0.0
    return {
        "sum_edge_weight": float(np.sum(weights)) if weights else 0.0,
        "max_edge_weight": float(np.max(weights)) if weights else 0.0,
        "subgraph_density": density,
        "n_drugs": len(drugs),
    }

emb_rows = []
graphstat_rows = []
for hadm_id, drugs in tqdm(hadm_drugs.items(), desc="Building patient-level graph features"):
    emb_rows.append({"hadm_id": hadm_id, **{f"emb_{i}": v for i, v in enumerate(patient_embedding_feature(drugs))}})
    graphstat_rows.append({"hadm_id": hadm_id, **patient_graph_theoretic_features(hadm_id, drugs)})

embedding_features_df = pd.DataFrame(emb_rows).set_index("hadm_id")
graphstat_features_df = pd.DataFrame(graphstat_rows).set_index("hadm_id")

print("Embedding feature matrix:", embedding_features_df.shape)
print("Graph-theoretic feature matrix:", graphstat_features_df.shape)


## 4. Rebuild tabular baseline (one small BigQuery query for comorbidity counts)

This is the only BigQuery call in this notebook -- a `COUNT(DISTINCT icd_code)`
aggregate over `diagnoses_icd`, filtered to the ~10k cached `hadm_id`s. Cheap
and fast. `WRITE_TRUNCATE` is used on the temp table (same fix applied in the
original local run, to avoid the duplicate-row join bug from before).


In [ ]:
hadm_ids = cohort_df["hadm_id"].unique().tolist()
hadm_id_df = pd.DataFrame({"hadm_id": hadm_ids})
temp_table_id = f"{PROJECT_ID}.temp_polypharm.resume_hadm_ids"

client.query("CREATE SCHEMA IF NOT EXISTS temp_polypharm").result()
load_job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
job = client.load_table_from_dataframe(hadm_id_df, temp_table_id, job_config=load_job_config)
job.result()

check_df = client.query(
    f"SELECT COUNT(*) AS n, COUNT(DISTINCT hadm_id) AS n_distinct FROM `{temp_table_id}`"
).to_dataframe()
assert int(check_df["n"][0]) == int(check_df["n_distinct"][0]) == len(hadm_id_df), "temp table dedup check failed"
print(f"Temp table verified: {len(hadm_id_df)} rows, all distinct.")

diag_query = f"""
SELECT hadm_id, COUNT(DISTINCT icd_code) AS n_diagnoses
FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd`
WHERE hadm_id IN (SELECT hadm_id FROM `{temp_table_id}`)
GROUP BY hadm_id
"""
diag_df = client.query(diag_query).to_dataframe().set_index("hadm_id")
print(f"diag_df: {diag_df.shape}")


In [ ]:
drug_onehot = pd.crosstab(rx_df["hadm_id"], rx_df["generic_drug"])
drug_onehot = (drug_onehot > 0).astype(int)

demo_df = cohort_df.set_index("hadm_id")[["anchor_age", "gender"]].copy()
demo_df["gender_male"] = (demo_df["gender"] == "M").astype(int)
demo_df = demo_df.drop(columns=["gender"])
demo_df = demo_df.join(diag_df, how="left").fillna({"n_diagnoses": 0})

tabular_features_df = demo_df.join(drug_onehot, how="left").fillna(0)
print("Tabular baseline feature matrix:", tabular_features_df.shape)


## 5. Delirium/tabular label-leakage fix

`delirium_label` is partly DEFINED by whether the patient received one of
`DELIRIUM_DRUG_KEYWORDS` (antipsychotics/benzodiazepines used for agitation).
The tabular baseline's one-hot drug indicators include columns for those exact
same drugs, so a tabular model can trivially reconstruct much of the label just
by reading "was haloperidol given" -- this produced an artificially inflated
AUROC (0.944 for xgboost) in the original local run. This is circularity from
the label's own definition, not genuine predictive signal, and must not be
reported as if it were.

We drop the delirium-defining drug columns from the tabular feature matrix ONLY
for `delirium_label` (AKI/bleeding labels are not drug-defined, so their
results from the local run are valid and left untouched), refit the 4
tabular models for delirium_label, and patch the corrected rows into the loaded
`model_results.csv`. **This exclusion must be reported explicitly in the
paper's Methods.**


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
import xgboost as xgb

DELIRIUM_DRUG_KEYWORDS = [
    "haloperidol", "quetiapine", "olanzapine", "risperidone", "ziprasidone",
    "lorazepam", "midazolam", "diazepam",
]

FEATURE_SETS = {
    "embedding": embedding_features_df,
    "graph_theoretic": graphstat_features_df,
    "tabular": tabular_features_df,
}
N_FOLDS = 5
N_BOOTSTRAP = 1000
RANDOM_STATE = 42

def get_models():
    return {
        "logistic_regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        "xgboost": xgb.XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
            device=XGB_DEVICE, tree_method="hist",
        ),
        "mlp": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=RANDOM_STATE),
    }

def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    stats = []
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            stats.append(metric_fn(y_true[idx], y_prob[idx]))
        except Exception:
            continue
    if not stats:
        return (np.nan, np.nan)
    return (np.percentile(stats, 2.5), np.percentile(stats, 97.5))

model_results_df = pd.read_csv("model_results.csv")
print(f"Loaded model_results.csv: {model_results_df.shape}")
print("Rows being replaced (leaky):")
print(model_results_df[(model_results_df.outcome == "delirium_label") & (model_results_df.feature_set == "tabular")])


In [ ]:
DELIRIUM_LEAKY_TABULAR_COLS = [
    c for c in tabular_features_df.columns
    if any(kw in c for kw in DELIRIUM_DRUG_KEYWORDS)
]
print(f"Excluding {len(DELIRIUM_LEAKY_TABULAR_COLS)} delirium-defining drug columns: {DELIRIUM_LEAKY_TABULAR_COLS}")

X_full_clean = tabular_features_df.drop(columns=DELIRIUM_LEAKY_TABULAR_COLS)
y = cohort_df.set_index("hadm_id").loc[:, "delirium_label"]
common_idx = X_full_clean.index.intersection(y.index)
X = X_full_clean.loc[common_idx].values
y_aligned = y.loc[common_idx].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

corrected_rows = []
for model_name in get_models():
    oof_pred = np.zeros(len(y_aligned))
    for train_idx, test_idx in skf.split(X_scaled, y_aligned):
        model = get_models()[model_name]
        model.fit(X_scaled[train_idx], y_aligned[train_idx])
        oof_pred[test_idx] = model.predict_proba(X_scaled[test_idx])[:, 1]

    auroc = roc_auc_score(y_aligned, oof_pred)
    auprc = average_precision_score(y_aligned, oof_pred)
    f1 = f1_score(y_aligned, (oof_pred >= 0.5).astype(int))
    brier = brier_score_loss(y_aligned, oof_pred)
    auroc_lo, auroc_hi = bootstrap_ci(y_aligned, oof_pred, roc_auc_score)

    corrected_rows.append({
        "outcome": "delirium_label", "feature_set": "tabular", "model": model_name,
        "n": len(y_aligned), "n_positive": int(y_aligned.sum()),
        "auroc": auroc, "auroc_ci_low": auroc_lo, "auroc_ci_high": auroc_hi,
        "auprc": auprc, "f1": f1, "brier": brier,
    })
    print(f"[CORRECTED] delirium_label | tabular (leak-free) | {model_name:20s} | "
          f"AUROC={auroc:.3f} [{auroc_lo:.3f},{auroc_hi:.3f}]  AUPRC={auprc:.3f}  F1={f1:.3f}  Brier={brier:.3f}")

corrected_df = pd.DataFrame(corrected_rows)

mask = (model_results_df.outcome == "delirium_label") & (model_results_df.feature_set == "tabular")
model_results_df = model_results_df[~mask]
model_results_df = pd.concat([model_results_df, corrected_df], ignore_index=True)

model_results_path = os.path.join(RESULTS_DIR, "model_results.csv")
model_results_df.to_csv(model_results_path, index=False)
save_manifest_entry(model_results_path, "Corrected model results grid (delirium/tabular refit without leaky drug columns)")
print("\nmodel_results.csv patched and saved.")


## 6. Interpretability

### Perturbation analysis for embedding models (batched, GPU-agnostic speedup)

The original local run called `model.predict_proba()` once per drug per
high-risk patient -- for ~1,000 high-risk patients x ~15-20 drugs each, that's
15,000-20,000 individual unbatched calls per outcome, which is what made this
stage slow locally (tree-ensemble models have real per-call overhead). This
version builds one big matrix of [base_vector, perturbed_vector_per_drug] for
ALL high-risk patients in an outcome and scores it with a single
`predict_proba()` call, then reshapes the results back per patient/drug. This
is the actual fix for the slowness -- it is a batching fix, not a GPU fix
(sklearn CPU inference on one big batch is already fast; the problem was call
count, not FLOPs).


In [ ]:
def best_model_per_outcome_feature_set(results_df, outcome, feature_set):
    sub = results_df[(results_df.outcome == outcome) & (results_df.feature_set == feature_set)]
    return sub.sort_values("auroc", ascending=False).iloc[0]["model"]

def fit_full_model(model_name, X, y):
    model = get_models()[model_name]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    model.fit(X_scaled, y)
    return model, scaler

# Minimum number of high-risk patients a drug must appear in before its mean
# perturbation delta is trusted. Without this, the ranking is dominated by
# idiosyncratic/rare drugs that only 1-2 patients happened to receive -- a mean
# of 1 sample is not a stable estimate of "risk contribution," it is noise that
# happened to land on a high-risk patient. Observed in the unfiltered run:
# 60-67% of the top-15 entries for AKI/bleeding had n_high_risk_patients == 1.
MIN_PATIENTS_FOR_RANKING = 10

for outcome in OUTCOMES:
    y = cohort_df.set_index("hadm_id").loc[:, outcome]
    common_idx = embedding_features_df.index.intersection(y.index)
    X_emb = embedding_features_df.loc[common_idx]
    y_aligned = y.loc[common_idx]

    best_model_name = best_model_per_outcome_feature_set(model_results_df, outcome, "embedding")
    model, scaler = fit_full_model(best_model_name, X_emb.values, y_aligned.values)

    X_scaled_full = scaler.transform(X_emb.values)
    base_pred = model.predict_proba(X_scaled_full)[:, 1]

    high_risk_thresh = np.percentile(base_pred, 90)
    high_risk_hadm = X_emb.index[base_pred >= high_risk_thresh]

    # --- Build one big batch: for each high-risk patient, one base row + one
    # perturbed row per drug in their list. Track (hadm_id, drug_removed) per row.
    batch_vectors = []
    batch_index = []  # (hadm_id, drug_or_None) -- None marks the base (unperturbed) row
    for hadm_id in high_risk_hadm:
        drugs = hadm_drugs.get(hadm_id, [])
        if len(drugs) < 2:
            continue
        batch_vectors.append(patient_embedding_feature(drugs))
        batch_index.append((hadm_id, None))
        for d in drugs:
            remaining = [x for x in drugs if x != d]
            batch_vectors.append(patient_embedding_feature(remaining))
            batch_index.append((hadm_id, d))

    batch_matrix = np.vstack(batch_vectors)
    batch_scaled = scaler.transform(batch_matrix)
    batch_scores = model.predict_proba(batch_scaled)[:, 1]
    print(f"{outcome}: scored {len(batch_scores)} perturbation rows in a single batched call "
          f"(vs. {len(batch_scores)} individual calls in the original unbatched version).")

    base_scores = {}
    drug_deltas = defaultdict(list)
    for (hadm_id, drug), score in zip(batch_index, batch_scores):
        if drug is None:
            base_scores[hadm_id] = score
    for (hadm_id, drug), score in zip(batch_index, batch_scores):
        if drug is not None:
            drug_deltas[drug].append(base_scores[hadm_id] - score)

    delta_summary = pd.DataFrame([
        {"drug": d, "mean_risk_contribution": np.mean(v), "n_high_risk_patients": len(v)}
        for d, v in drug_deltas.items()
    ]).sort_values("mean_risk_contribution", ascending=False)

    n_before_filter = len(delta_summary)
    reliable = delta_summary[delta_summary["n_high_risk_patients"] >= MIN_PATIENTS_FOR_RANKING]
    n_excluded = n_before_filter - len(reliable)
    print(f"{outcome}: excluded {n_excluded}/{n_before_filter} drugs with "
          f"fewer than {MIN_PATIENTS_FOR_RANKING} high-risk patients (unstable mean) before ranking.")
    if len(reliable) < 15:
        print(f"  WARNING: only {len(reliable)} drugs meet the reliability threshold "
              f"for {outcome} -- top list below has fewer than 15 rows as a result.")

    top15 = reliable.head(15)
    out_path = os.path.join(RESULTS_DIR, f"top_risk_drugs_{outcome}.csv")
    top15.to_csv(out_path, index=False)
    save_manifest_entry(out_path, f"Top-{len(top15)} perturbation-ranked risk-associated drugs for {outcome} "
                                   f"(best model: {best_model_name}; min {MIN_PATIENTS_FOR_RANKING} patients/drug)")
    print(f"\n=== {outcome} (best embedding model: {best_model_name}) ===")
    print(top15.to_string(index=False))


### SHAP on the tabular baseline (best model per outcome, tree-based only)

In [ ]:
import shap

for outcome in OUTCOMES:
    y = cohort_df.set_index("hadm_id").loc[:, outcome]
    X_tab_full = tabular_features_df.drop(columns=DELIRIUM_LEAKY_TABULAR_COLS) if outcome == "delirium_label" else tabular_features_df
    common_idx = X_tab_full.index.intersection(y.index)
    X_tab = X_tab_full.loc[common_idx]
    y_aligned = y.loc[common_idx]

    best_model_name = best_model_per_outcome_feature_set(model_results_df, outcome, "tabular")
    if best_model_name not in ("random_forest", "xgboost"):
        print(f"Skipping SHAP tree-explainer for {outcome}: best tabular model was {best_model_name} (not tree-based).")
        continue

    model, scaler = fit_full_model(best_model_name, X_tab.values, y_aligned.values)
    X_scaled_full = scaler.transform(X_tab.values)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_scaled_full[:2000])  # cap for runtime

    shap_summary_path = os.path.join(RESULTS_DIR, f"shap_summary_tabular_{outcome}.pkl")
    with open(shap_summary_path, "wb") as f:
        pickle.dump({
            "shap_values": shap_values,
            "feature_names": X_tab.columns.tolist(),
            "model_name": best_model_name,
        }, f)
    save_manifest_entry(shap_summary_path, f"SHAP values for best tabular model ({best_model_name}) on {outcome}")
    print(f"Saved SHAP summary for {outcome} ({best_model_name}).")


## 7. External sanity check (manual, post-hoc)

Same as the original notebook: no external DDI API is called. Manually
cross-reference these against Lexicomp/Micromedex/FDA labels before claiming
clinical validity in the Discussion section.


In [ ]:
for outcome in OUTCOMES:
    path = os.path.join(RESULTS_DIR, f"top_risk_drugs_{outcome}.csv")
    if os.path.exists(path):
        print(f"--- Top risk-associated drugs for {outcome} (manually cross-check against DDI references) ---")
        print(pd.read_csv(path).to_string(index=False))
        print()


## 8. Save everything and download

In [ ]:
manifest_path = os.path.join(RESULTS_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump(MANIFEST, f, indent=2)

zip_path = "results_resumed.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(RESULTS_DIR):
        for fn in fnames:
            full_path = os.path.join(root, fn)
            zf.write(full_path, arcname=os.path.relpath(full_path, RESULTS_DIR))

print(f"Results zipped to {zip_path}\n")
print(f"{'FILE':45s} {'SIZE':>10s}  DESCRIPTION")
print("-" * 100)
for entry in MANIFEST:
    fp = entry["file"]
    size = os.path.getsize(fp) if os.path.exists(fp) else 0
    size_str = f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.1f} MB"
    print(f"{os.path.basename(fp):45s} {size_str:>10s}  {entry['description']}")

print(f"\nGenerated: {datetime.now().isoformat()}")
print("\nNOTE: merge this zip's contents with your original local results/ folder --")
print("this run only produced the NEW files (corrected model_results.csv, top_risk_drugs_*.csv,")
print("shap_summary_*.pkl, manifest.json). The graph/embedding/cohort files from the local run")
print("are still needed for Phase 2 (paper writing) and are not re-saved here.")


In [ ]:
from google.colab import files
files.download(zip_path)
